In [ ]:
%%capture 
# Import necessary libraries
import os, mne, re, csv, numpy as np, matplotlib.pyplot as plt
from glob import glob
from mne.channels import make_standard_montage
from eeg_function import drop_trials, single_trial_normalisation
# Set the derivatives directory path
root_derivatives = 'C:/Users/mfbpe/Desktop/DATA/2025_Valuation/derivatives/'

# Change the current working directory to the derivatives directory
os.chdir(root_derivatives)

all_files_path = sorted(glob(f"*raw.fif"), key=len)

datafile=open("data_ERP.csv","w", newline="")
writer=csv.writer(datafile, delimiter=";")
writer.writerow(["Participant", "reward", "difficulty", "Trial_ERP", "Outlier", "mean_N200", "mean_P300", "peak_N200", "mean_to_mean_P300"])

event_id_stim = { 
    'stim/motiv/extr':11,'stim/motiv/hard':12,'stim/motiv/easy':13,'stim/amotiv/extr':14,'stim/amotiv/hard':15,'stim/amotiv/easy':16
}


for i,part in enumerate(all_files_path[:20]): #only first 20 participants for the sake of practice

    raw = mne.io.read_raw_fif(part, preload=True)
    
    events = mne.find_events(raw, shortest_event=1)            

    epochs= mne.Epochs(raw, events,baseline = (-.3, -.1), event_id=event_id_stim, picks='eeg',
            tmin=-1.6, tmax=2,preload=True, detrend=None, on_missing='ignore')

    if i==0:  
        sphere = mne.make_sphere_model('auto', 'auto', epochs.info)
        src = mne.setup_volume_source_space(sphere=sphere)
        forward = mne.make_forward_solution(epochs.info, trans=None, src=src, bem=sphere)

    epochs.set_eeg_reference('REST', forward=forward)
    

    chs_index = [i for i,x in enumerate(epochs.info['ch_names']) if x in ['FCz','Pz']]
            # Combine channels into one average

    epochs_comb = mne.channels.combine_channels(epochs, dict(Avg=chs_index))
    _,list_ti,_,_,_ = drop_trials(epochs_comb, do_mean=1, do_peak=1, do_slope=1, T1=-.3,T2=.5, chs='all')  
    
    epochs.drop(list_ti==0)

    chs_index = [i for i,x in enumerate(epochs.info['ch_names']) if x in ['FCz','Pz']]
    epochs_N200 = mne.channels.combine_channels(epochs, dict(Avg=chs_index))

    chs_index = [i for i,x in enumerate(epochs.info['ch_names']) if x in ['Pz','CPz']]
    epochs_P300 = mne.channels.combine_channels(epochs, dict(Avg=chs_index))

    tmin_N200, tmax_N200 = .240, .290
    tmin_P300, tmax_P300 = .325, .375
    
    list_trigger_condition_level1 = [list(event_id_stim.keys())[list(event_id_stim.values()).index(x)].split("/")[1] for x in epochs.events[:,2]]
    list_trigger_condition_level2 = [list(event_id_stim.keys())[list(event_id_stim.values()).index(x)].split("/")[2] for x in epochs.events[:,2]]


    for ite_trial in range(len(epochs)):
        N200_mean = np.mean(epochs_N200.get_data(units="uV", tmin=tmin_N200, tmax=tmax_N200), axis=2)[ite_trial][0]
        P300_mean = np.mean(epochs_P300.get_data(units="uV", tmin=tmin_P300, tmax=tmax_P300), axis=2)[ite_trial][0]

        N200_peak = np.max(epochs_N200.get_data(units="uV", tmin=tmin_N200, tmax=tmax_N200), axis=2)[ite_trial][0]

        P300_mean_to_mean = P300_mean - np.mean(epochs_P300.get_data(units="uV", tmin=tmin_N200, tmax=tmax_N200), axis=2)[ite_trial][0] # minue N200

        writer.writerow([part,  list_trigger_condition_level1[ite_trial], list_trigger_condition_level2[ite_trial],ite_trial, list_ti[ite_trial], round(N200_mean,3), round(P300_mean,3), round(N200_peak,3), round(P300_mean_to_mean,3)])

    datafile.flush()
datafile.close()
